This cell initializes the analysis by loading pre-computed data artifacts from previous processing steps, which include various dataframes and embeddings related to public dialogues about AI. By avoiding the need to re-run resource-intensive operations like API calls or clustering algorithms, it streamlines the workflow and ensures that subsequent analyses can focus on comparing AI-related dialogues with non-AI dialogues, particularly in terms of concern and benefit framings. This efficiency is crucial for maintaining the integrity of the analysis while allowing for a deeper exploration of the distinctive characteristics of AI discourse.

In [ ]:
# @title Load pre-computed artifacts
# Run this cell before any analysis cell.  It loads all outputs written by
# 01_processing.ipynb from disk so this notebook never calls the OpenAI API
# or re-runs k-means.
from pathlib import Path
import pub_dialogue.utils as du
from pub_dialogue.utils import show_status, show_complete
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

OUTPUT_FOLDER     = Path("outputs")
CHECKPOINT_FOLDER = Path("checkpoints")

a = du.load_artifacts(OUTPUT_FOLDER, CHECKPOINT_FOLDER)

chunks_df    = a["chunks_df"]
concerns_df  = a["concerns_df"]
benefits_df  = a["benefits_df"]

concern_embeddings  = a["concern_embeddings"]
benefit_embeddings  = a["benefit_embeddings"]
concern_centroids   = a["concern_centroids"]
benefit_centroids   = a["benefit_centroids"]

concern_ids          = a["concern_ids"]
benefit_ids          = a["benefit_ids"]
cluster_labels       = a["cluster_labels"]
benefit_cluster_labels = a["benefit_cluster_labels"]
cluster_summary_df   = a["cluster_summary_df"]
benefit_cluster_summary_df = a["benefit_cluster_summary_df"]

framing_lens_mappings         = a["framing_lens_mappings"]
benefit_framing_lens_mappings = a["benefit_framing_lens_mappings"]

cluster_entropy           = a["cluster_entropy"]
cluster_entropy_norm      = a["cluster_entropy_norm"]
cross_cutting_clusters    = a["cross_cutting_clusters"]

benefit_cluster_entropy          = a["benefit_cluster_entropy"]
normalized_entropy_benefits      = a["normalized_entropy_benefits"]
cross_cutting_clusters_benefits  = a["cross_cutting_clusters_benefits"]

# Convenience: CLUSTER_LABELS / BENEFIT_CLUSTER_LABELS dicts used by plots
CLUSTER_LABELS        = {int(k): v for k, v in cluster_labels.items()}
BENEFIT_CLUSTER_LABELS = {int(k): v for k, v in benefit_cluster_labels.items()}

# Uppercase aliases expected by analysis cells
FRAMING_LENS_MAPPINGS        = framing_lens_mappings
BENEFIT_FRAMING_LENS_MAPPINGS = benefit_framing_lens_mappings

# Cross-cutting label dicts referenced by analysis cells
cross_cutting_labels = {
    cid: cluster_labels[str(cid)]
    for cid in (cross_cutting_clusters or [])
    if str(cid) in cluster_labels
}
cross_cutting_labels_benefits = {
    cid: benefit_cluster_labels[str(cid)]
    for cid in (cross_cutting_clusters_benefits or [])
    if str(cid) in benefit_cluster_labels
}

# Re-apply technology_meta merge if not already present in the loaded CSV
if "technology_meta" not in concerns_df.columns:
    _tech = chunks_df[["chunk_id", "technology_meta"]]
    concerns_df = concerns_df.merge(_tech, on="chunk_id", how="left")
    benefits_df = benefits_df.merge(_tech, on="chunk_id", how="left")

print(f"Artifacts loaded from {OUTPUT_FOLDER} / {CHECKPOINT_FOLDER}")
print(f"  Chunks: {len(chunks_df):,}  |  Concerns: {len(concerns_df):,}  |  Benefits: {len(benefits_df):,}")

This cell computes and visualizes the distinct framing profiles of AI-related dialogues compared to non-AI dialogues, focusing on various concern and benefit framings. By analyzing the salience of different framing lenses across technologies, it provides insights into how public perceptions of AI differ from those of other technologies, which is crucial for understanding the unique challenges and opportunities in AI discourse. The results are presented in a radar plot and saved for further analysis, contributing to the overall goal of identifying distinctive features in AI public dialogues.

In [ ]:
# @title Compare AI and non-AI dialogues by framing lens

show_status("Computing AI vs non-AI framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    lens_salience_by_tech[tech] = {}
    for lens_name, data in FRAMING_LENS_MAPPINGS.items():
        lens_mask = concerns_df['cluster_id'].isin(data['cluster_ids'])
        lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

lens_salience_df = pd.DataFrame(lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
non_ai_avg = lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in lens_salience_df.index and len(non_ai_avg) > 0:
    categories = list(non_ai_avg.index)
    ai_vals = lens_salience_df.loc['AI', categories].tolist()
    base_vals = non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_framing_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "ai_vs_nonai_framing_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

# Save table
lens_salience_df.to_csv(OUTPUT_FOLDER / "lens_salience_by_technology_meta.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI framing profile computed")


This cell computes a document-weighted baseline for non-AI dialogues to provide a more accurate comparison against AI dialogues regarding concern framings. By calculating the proportion of concern phrases across different lenses based on the total number of non-AI phrases, it addresses potential biases introduced by technology-weighted baselines, particularly the overrepresentation of larger non-AI categories. This diagnostic step is crucial for validating the distinctiveness of AI-related discussions, as it allows for a clearer understanding of how AI dialogues differ from non-AI ones across various thematic lenses.

In [ ]:
# @title (v16) Document-weighted AI vs non-AI baseline (concerns) — diagnostic
# Issue 10 partial fix from the audit: the canonical AI vs non-AI comparison
# uses a TECHNOLOGY-WEIGHTED baseline (each non-AI technology contributes
# equally to the baseline), which gives small-N technologies like Quantum
# (6 phrases) the same weight as Genetic technology (675 phrases). This is
# the right move conceptually for "what does the average non-AI dialogue
# look like", but it can produce noisy baselines.
#
# This cell computes a DOCUMENT-WEIGHTED baseline as a sanity check: the
# baseline is simply "what proportion of all non-AI concern phrases fall in
# each lens?". This is dominated by Genetic technology, which is the
# largest non-AI corpus by volume. Compare this side-by-side with the
# technology-weighted baseline above; if conclusions about AI distinctiveness
# only hold under one weighting, that is a finding worth reporting.

show_status("Computing document-weighted non-AI baseline...")

ai_mask = concerns_df["technology_meta"] == "AI"
non_ai_mask = ~ai_mask

# Document-weighted: pool all non-AI phrases, count lens shares directly
non_ai_lens_counts = {}
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df["cluster_id"].isin(data["cluster_ids"])
    non_ai_lens_counts[lens_name] = int((non_ai_mask & lens_mask).sum())

non_ai_total = int(non_ai_mask.sum())
non_ai_doc_weighted = {k: v / non_ai_total for k, v in non_ai_lens_counts.items()} if non_ai_total else {}

# AI shares (unchanged)
ai_lens_shares = {}
ai_total = int(ai_mask.sum())
for lens_name, data in FRAMING_LENS_MAPPINGS.items():
    lens_mask = concerns_df["cluster_id"].isin(data["cluster_ids"])
    ai_lens_shares[lens_name] = float((ai_mask & lens_mask).sum() / ai_total) if ai_total else 0.0

# Combine
doc_weighted_comparison = pd.DataFrame({
    "lens": list(FRAMING_LENS_MAPPINGS.keys()),
    "ai_share": [ai_lens_shares[l] for l in FRAMING_LENS_MAPPINGS],
    "non_ai_doc_weighted_share": [non_ai_doc_weighted.get(l, 0.0) for l in FRAMING_LENS_MAPPINGS],
})
doc_weighted_comparison["ai_minus_doc_weighted"] = (
    doc_weighted_comparison["ai_share"] - doc_weighted_comparison["non_ai_doc_weighted_share"]
)
doc_weighted_comparison = doc_weighted_comparison.sort_values("ai_minus_doc_weighted", ascending=False)

doc_weighted_comparison.to_csv(OUTPUT_FOLDER / "ai_vs_nonai_lens_doc_weighted.csv", index=False)

print("AI vs non-AI by lens (document-weighted baseline):")
print(doc_weighted_comparison.to_string(index=False))
print()
print("Compare with the technology-weighted version above. If the rank order")
print("of AI-distinctive lenses changes substantially between the two, the")
print("AI distinctiveness claim is sensitive to baseline construction.")

show_complete("Document-weighted baseline complete")


This cell computes and compares the concern profiles of AI and non-AI dialogues, focusing on cross-cutting themes identified in previous analyses. By normalizing the counts of concerns across different technologies, it generates a comprehensive view of how public dialogues about AI differ from those concerning other technologies, which is crucial for understanding the unique framing of AI-related issues in public discourse. The results are saved for further analysis, allowing researchers to explore the implications of these distinct concern profiles on public perception and policy discussions surrounding AI.

In [ ]:
from pub_dialogue.utils import show_status, show_complete


This cell analyzes and compares the concern profiles associated with AI and non-AI dialogues by focusing on cross-cutting themes identified in prior sections. By calculating the distribution of concerns across different technologies, it highlights how AI-related discussions may differ from those surrounding other technologies, providing insights into public sentiment and framing that are crucial for understanding the societal implications of AI. The results are saved for further analysis, enabling a deeper exploration of the unique characteristics of AI dialogues in public discourse.

In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting concerns

show_status("Computing AI vs non-AI concern profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels from SECTION 3 (derived from entropy threshold)
cross_cutting_cluster_ids = set(cross_cutting_labels.keys())

# Technology profiles over cross-cutting clusters
profiles = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask & concerns_df['cluster_id'].isin(cross_cutting_cluster_ids), 'cluster_id'].value_counts()
    # normalise over all concerns for that tech (so "attention share" to cross-cutting clusters)
    profiles[tech] = (counts / tech_total).to_dict()

profiles_df = pd.DataFrame(profiles).T.fillna(0)

non_ai_avg = profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Save for later use
profiles_df.to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profiles_by_tech.csv")
non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting concern profiles computed")


This code snippet constructs a dictionary that maps cluster identifiers to their corresponding labels, contingent on the existence of a summary DataFrame containing the necessary columns. This mapping is crucial for visualizing and interpreting the results of the clustering analysis, as it allows for a clearer understanding of the distinct themes and concerns present in AI-related dialogues compared to non-AI dialogues, thereby enhancing the overall analysis of public sentiment and framing in discussions about AI.

In [ ]:
# --- Build CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "summary_df" in globals() and {"cluster_id", "label"}.issubset(summary_df.columns):
    CLUSTER_LABELS = dict(zip(summary_df["cluster_id"], summary_df["label"]))
else:
    CLUSTER_LABELS = {}


This cell generates a radar chart that visually compares the concern framings associated with AI dialogues against a baseline of non-AI dialogues. By identifying and highlighting the most significant differences in concern categories, this visualization helps to elucidate the unique aspects of public sentiment regarding AI, which is crucial for understanding how these dialogues shape perceptions and policy discussions surrounding technology. The resulting figure is saved in both HTML and PNG formats for further analysis and presentation.

In [ ]:
# @title Visualise AI concern fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in profiles_df.index and len(non_ai_avg) > 0:
    show_status("Creating AI fingerprint radar (AI vs non-AI)...")

    diff = (profiles_df.loc['AI'] - non_ai_avg).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = profiles_df.loc['AI', selected].tolist()
    base_vals = non_ai_avg[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI concern fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "ai_vs_nonai_concern_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "ai_vs_nonai_concern_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

show_complete("AI fingerprint radar created")


This cell computes and identifies the distinctive concerns and framings associated with AI dialogues compared to non-AI technologies by analyzing cluster profiles and framing lenses. By calculating the differences in concern and framing salience between AI and other technologies, it highlights the unique aspects of public discourse surrounding AI, which is crucial for understanding societal perceptions and guiding effective communication strategies in AI-related discussions. The results are saved for further analysis, providing valuable insights into how AI is framed in public dialogues.

In [ ]:
# @title Compute per-technology concern cluster profiles (all clusters)
# salience_df: rows = technologies, columns = cluster_ids
# Each value is the share of that technology's concerns in the given cluster.

TECH_COL = 'technology_meta'
technologies = sorted(concerns_df[TECH_COL].dropna().unique().tolist())

cluster_salience_by_tech = {}
for tech in technologies:
    tech_mask = concerns_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = concerns_df.loc[tech_mask, 'cluster_id'].value_counts()
    cluster_salience_by_tech[tech] = (counts / tech_total).to_dict()

salience_df = pd.DataFrame(cluster_salience_by_tech).T.fillna(0)
salience_df.columns = salience_df.columns.astype(int)

print(f"salience_df: {salience_df.shape}  (technologies x concern clusters)")


This cell analyzes and identifies the unique concerns and framings present in AI-related public dialogues compared to non-AI dialogues by computing the differences in cluster profiles and framing lenses. By assessing the distinctive features of AI discussions, the analysis highlights specific areas of concern and thematic emphasis that may influence public perception and policy-making, ultimately contributing to a deeper understanding of how AI is uniquely positioned in societal discourse. The results are saved for further exploration and validation, making them a valuable resource for researchers in NLP and public dialogue studies.

In [ ]:
# @title Identify distinctive AI concerns and framings

show_status("Computing distinctive concerns and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive concern clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = salience_df  # from 6.1a (rows=tech, cols=cluster_id)

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_concerns = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_concerns)
    distinctive_concerns.to_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv", index=False)

# Distinctive framings (lens level)
if 'AI' in lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")


This cell imports the `pretty_label` function from the `dialogue_utils` module, which is likely designed to enhance the presentation of dialogue data by applying consistent and informative labeling. This functionality is crucial for the subsequent analysis, as it ensures that the comparisons between AI-related and non-AI dialogues are clearly articulated and visually accessible, thereby facilitating a deeper understanding of the distinctiveness of AI discourse in public dialogues.

In [ ]:
# pretty_label — imported from dialogue_utils (see Cell 5)
pass

This cell computes and visualizes the differences in benefit framings between AI-related and non-AI public dialogues, providing a technology-weighted baseline for comparison. By analyzing the salience of various benefit lenses across different technologies, it highlights how AI discussions may uniquely frame benefits compared to other technologies, which is crucial for understanding public perceptions and concerns about AI in the broader context of technological discourse. The resulting radar plot and data tables facilitate further analysis and interpretation of these distinct framing patterns.

In [ ]:
# @title Compare AI and non-AI dialogues by framing lens (benefits)

show_status("Computing AI vs non-AI benefit framing profile (technology-weighted baseline)...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Lens salience per technology (within-technology proportions)
benefit_lens_salience_by_tech = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    benefit_lens_salience_by_tech[tech] = {}
    for lens_name, data in BENEFIT_FRAMING_LENS_MAPPINGS.items():
        lens_mask = benefits_df['cluster_id'].isin(data['cluster_ids'])
        benefit_lens_salience_by_tech[tech][lens_name] = float((tech_mask & lens_mask).sum() / tech_total)

benefit_lens_salience_df = pd.DataFrame(benefit_lens_salience_by_tech).T.fillna(0)

# Technology-weighted non-AI baseline (each non-AI technology counts equally)
benefit_non_ai_avg = benefit_lens_salience_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

# Plot radar: AI vs non-AI avg
if 'AI' in benefit_lens_salience_df.index and len(benefit_non_ai_avg) > 0:
    categories = list(benefit_non_ai_avg.index)
    ai_vals = benefit_lens_salience_df.loc['AI', categories].tolist()
    base_vals = benefit_non_ai_avg[categories].tolist()

    # Close the loop
    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit framing profile vs non‑AI baseline (technology‑weighted)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_framing_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "benefit_ai_vs_nonai_framing_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

# Save table
benefit_lens_salience_df.to_csv(OUTPUT_FOLDER / "benefit_lens_salience_by_technology_meta.csv")
benefit_non_ai_avg.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "benefit_non_ai_avg_framing_profile.csv")
show_complete("AI vs non-AI benefit framing profile computed")


This cell analyzes and compares the benefit framings present in AI-related dialogues against those in non-AI dialogues by calculating the normalized benefit profiles across identified cross-cutting clusters. By aggregating and averaging the benefit distributions for different technologies, it generates a detailed overview of how public discourse on AI differs from other technologies, which is crucial for understanding the unique societal implications and perceptions surrounding AI. The results are saved for further examination, contributing to the broader analysis of distinctive features in public dialogues about AI.

In [ ]:
# @title Compare AI and non-AI dialogues on cross-cutting benefit clusters

show_status("Computing AI vs non-AI benefit profile on cross-cutting clusters...")

TECH_COL = 'technology_meta'
technologies = sorted(benefits_df[TECH_COL].dropna().unique().tolist())
non_ai_techs = [t for t in technologies if t != 'AI']

# Use cross_cutting_labels_benefits from cell 60 (derived from entropy threshold)
cross_cutting_cluster_ids_benefits = set(cross_cutting_labels_benefits.keys())

# Technology profiles over cross-cutting clusters
benefit_profiles = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask & benefits_df['cluster_id'].isin(cross_cutting_cluster_ids_benefits), 'cluster_id'].value_counts()
    # normalise over all benefits for that tech (so "attention share" to cross-cutting clusters)
    benefit_profiles[tech] = (counts / tech_total).to_dict()

benefit_profiles_df = pd.DataFrame(benefit_profiles).T.fillna(0)

benefit_non_ai_avg_cluster = benefit_profiles_df.loc[non_ai_techs].mean(axis=0) if len(non_ai_techs) else pd.Series(dtype=float)

benefit_profiles_df.to_csv(OUTPUT_FOLDER / "benefit_cross_cutting_cluster_profiles_by_tech.csv")
benefit_non_ai_avg_cluster.to_frame('non_ai_avg_tech_weighted').to_csv(OUTPUT_FOLDER / "benefit_cross_cutting_cluster_profile_non_ai_avg.csv")

show_complete("Cross-cutting benefit profiles computed")


This code snippet constructs a dictionary that maps cluster IDs to their corresponding labels for benefit framings in AI-related dialogues. By organizing this information, it facilitates the visualization and comparison of how different benefits are articulated in AI versus non-AI discussions, which is crucial for understanding the distinctiveness of public sentiment and framing around AI topics.

In [ ]:
# --- Build BENEFIT_CLUSTER_LABELS dict for plots (cluster_id -> label) ---
if "benefit_summary_df" in globals() and {"cluster_id", "label"}.issubset(benefit_summary_df.columns):
    BENEFIT_CLUSTER_LABELS = dict(zip(benefit_summary_df["cluster_id"], benefit_summary_df["label"]))
else:
    BENEFIT_CLUSTER_LABELS = {}


This cell generates a radar chart that visually compares the benefit framings of AI dialogues against a baseline of non-AI dialogues, highlighting the most significant differences in their respective benefit profiles. By identifying and displaying the top and bottom benefit clusters, this visualization aids in understanding how public perceptions of AI differ from those of other technologies, which is crucial for informing policy and communication strategies in AI discourse. The resulting chart serves as a valuable tool for researchers and practitioners aiming to navigate the complexities of public sentiment surrounding AI.

In [ ]:
# @title Visualise AI benefit fingerprint (AI vs non-AI)

TECH_COL = 'technology_meta'

if 'AI' in benefit_profiles_df.index and len(benefit_non_ai_avg_cluster) > 0:
    show_status("Creating AI benefit fingerprint radar (AI vs non-AI)...")

    diff = (benefit_profiles_df.loc['AI'] - benefit_non_ai_avg_cluster).sort_values()
    top_low = diff.head(5).index.tolist()
    top_high = diff.tail(7).index.tolist()
    selected = top_low + top_high

    # Human-friendly labels
    def pretty_cluster(cid):
        return BENEFIT_CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    categories = [pretty_cluster(c) for c in selected]
    ai_vals = benefit_profiles_df.loc['AI', selected].tolist()
    base_vals = benefit_non_ai_avg_cluster[selected].tolist()

    categories_loop = categories + [categories[0]]
    ai_loop = ai_vals + [ai_vals[0]]
    base_loop = base_vals + [base_vals[0]]

    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=ai_loop, theta=categories_loop, fill='toself', name='AI'))
    fig.add_trace(go.Scatterpolar(r=base_loop, theta=categories_loop, fill='toself', name='Non‑AI average (tech‑weighted)'))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, max(ai_loop+base_loop)*1.1])),
        title="AI benefit fingerprint vs non‑AI baseline (cross‑cutting clusters)"
    )
    fig.write_html(OUTPUT_FOLDER / "benefit_ai_vs_nonai_radar.html")
# v18: also save a static PNG copy for paper use (requires kaleido package)
try:
    fig.write_image(OUTPUT_FOLDER / "benefit_ai_vs_nonai_radar.png", width=900, height=700, scale=2)
except Exception as _e:
    print(f"Note: PNG export failed ({_e}); HTML version still saved.")
    print("If you need PNG, install kaleido: !pip install -q kaleido")
    fig.show()

show_complete("AI benefit fingerprint radar created")


This cell computes and identifies the distinctive benefits and framings associated with AI compared to other technologies by analyzing benefit salience data. It calculates the differences in benefit clusters and framing lenses between AI and non-AI technologies, highlighting the unique aspects of AI-related dialogues. This analysis is crucial for understanding how public perceptions of AI differ from other technologies, informing future discussions and policy-making in the field of AI and public dialogue.

In [ ]:
# @title Compute per-technology benefit cluster profiles (all clusters)
# benefit_salience_df: rows = technologies, columns = cluster_ids

benefit_cluster_salience_by_tech = {}
for tech in technologies:
    tech_mask = benefits_df[TECH_COL] == tech
    tech_total = tech_mask.sum()
    if tech_total == 0:
        continue
    counts = benefits_df.loc[tech_mask, 'cluster_id'].value_counts()
    benefit_cluster_salience_by_tech[tech] = (counts / tech_total).to_dict()

benefit_salience_df = pd.DataFrame(benefit_cluster_salience_by_tech).T.fillna(0)
benefit_salience_df.columns = benefit_salience_df.columns.astype(int)

print(f"benefit_salience_df: {benefit_salience_df.shape}  (technologies x benefit clusters)")


This cell analyzes and quantifies the unique benefits and framings associated with AI technologies compared to other non-AI technologies. By calculating the differences in benefit clusters and framing lenses, it identifies which aspects of AI are perceived as more or less salient, providing insights into public perceptions and concerns that are critical for understanding the societal implications of AI. The results are saved for further exploration, contributing to a nuanced understanding of how AI dialogues differ from those surrounding other technologies.

In [ ]:
# @title Identify distinctive AI benefits and framings

show_status("Computing distinctive benefits and framings for AI...")

TECH_COL = 'technology_meta'
non_ai_techs = [t for t in technologies if t != 'AI']

# Distinctive benefit clusters (all clusters, tech-weighted baseline)
all_cluster_profiles = benefit_salience_df  # rows=tech, cols=cluster_id

if 'AI' in all_cluster_profiles.index and len(non_ai_techs) > 0:
    non_ai_avg_all = all_cluster_profiles.loc[non_ai_techs].mean(axis=0)
    ai_vs_base = (all_cluster_profiles.loc['AI'] - non_ai_avg_all).sort_values()

    most_under = ai_vs_base.head(10)
    most_over = ai_vs_base.tail(10)

    def label(cid):
        return BENEFIT_CLUSTER_LABELS.get(cid, f"Cluster {cid}")

    distinctive_benefits = pd.DataFrame({
        'cluster_id': list(most_under.index) + list(most_over.index),
        'cluster_label': [label(c) for c in list(most_under.index) + list(most_over.index)],
        'ai_minus_nonai': list(most_under.values) + list(most_over.values),
        'ai_share': [float(all_cluster_profiles.loc['AI', c]) for c in list(most_under.index) + list(most_over.index)],
        'nonai_share': [float(non_ai_avg_all[c]) for c in list(most_under.index) + list(most_over.index)],
    }).sort_values('ai_minus_nonai')

    display(distinctive_benefits)
    distinctive_benefits.to_csv(OUTPUT_FOLDER / "ai_distinctive_benefits.csv", index=False)

# Distinctive framings (lens level) — use benefit_lens_salience_df
if 'AI' in benefit_lens_salience_df.index and len(non_ai_techs) > 0:
    non_ai_avg_lens = benefit_lens_salience_df.loc[non_ai_techs].mean(axis=0)
    lens_diff = (benefit_lens_salience_df.loc['AI'] - non_ai_avg_lens).sort_values()

    df = pd.DataFrame({
        'framing_lens': lens_diff.index,
        'ai_minus_nonai': lens_diff.values,
        'ai_share': benefit_lens_salience_df.loc['AI', lens_diff.index].values,
        'nonai_share': non_ai_avg_lens[lens_diff.index].values
    }).sort_values('ai_minus_nonai')

    display(df.head(10))
    display(df.tail(10))
    df.to_csv(OUTPUT_FOLDER / "benefit_ai_distinctive_framings.csv", index=False)

show_complete("Distinctiveness outputs saved")


This cell imports the `pretty_label` function from the `dialogue_utils` module, which is likely designed to enhance the presentation of labels associated with dialogue data. This functionality is crucial for the subsequent analysis, as it will help in visually distinguishing and interpreting the various concern and benefit framings in AI-related dialogues compared to non-AI dialogues, ultimately contributing to a clearer understanding of their distinctive characteristics.

In [ ]:
# pretty_label — imported from dialogue_utils (see Cell 5)
pass

This cell compiles and organizes data on the top 20 cross-cutting concern clusters related to AI dialogues by merging two datasets: one containing shared concerns and another summarizing cluster characteristics. By generating a table that includes cluster labels, sizes, and prevalence statistics, it highlights the most significant concerns in AI discussions, which is crucial for understanding the distinctive themes and framing strategies in public dialogues about AI compared to other topics. This analysis aids in identifying patterns and potential areas of public apprehension or interest, informing future research and dialogue strategies.

In [ ]:
# @title (paper) Table 1: Top 20 cross-cutting concern clusters
# Joins shared_concerns_top20.csv with cluster_summary.csv to get labels
# alongside the shared-concern statistics.

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_sc = pd.read_csv(OUTPUT_FOLDER / "shared_concerns_top20.csv")
_cs = pd.read_csv(OUTPUT_FOLDER / "cluster_summary.csv")

_t1 = _sc.merge(_cs[["cluster_id", "label", "size"]], on="cluster_id", how="left")
_t1 = _t1[["cluster_id", "label", "size", "global_prevalence", "tech_entropy"]]
_t1 = _t1.rename(columns={
    "cluster_id": "Cluster ID",
    "label": "Cluster label",
    "size": "N phrases",
    "global_prevalence": "Share of all concern phrases",
    "tech_entropy": "Technology entropy (raw)",
})
_t1["Share of all concern phrases"] = (_t1["Share of all concern phrases"] * 100).round(2).astype(str) + "%"
_t1["Technology entropy (raw)"] = _t1["Technology entropy (raw)"].round(3)

print("=== Table 1: Top 20 cross-cutting concern clusters ===\n")
print(_t1.to_string(index=False))
_total_share = _sc["global_prevalence"].sum() * 100
print(f"\nTop 20 share = {_total_share:.1f}% of all concern phrases.")

_t1.to_csv(PAPER_ASSETS / "table1_top_cross_cutting_concerns.csv", index=False)
_t1.to_excel(PAPER_ASSETS / "table1_top_cross_cutting_concerns.xlsx", index=False)
print(f"Saved: {PAPER_ASSETS / 'table1_top_cross_cutting_concerns.xlsx'}")


This cell generates a two-panel table that compares the distinctiveness of AI-related dialogues against non-AI dialogues by analyzing cluster-level and lens-level data. By identifying over-indexed and under-indexed concerns and framings, it provides insights into how public perceptions of AI differ from other topics, which is crucial for understanding the unique narratives and themes that emerge in AI discourse and informing future research and policy discussions.

In [ ]:
# @title (paper) Table 2: AI distinctiveness (clusters and lenses)
# Two-panel table. Panel A: cluster-level top 10 over-indexed and top 10
# under-indexed. Panel B: lens-level distinctiveness under both baselines.

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_clusters = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv")
_lenses_tech = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_framings.csv")
_lenses_doc = pd.read_csv(OUTPUT_FOLDER / "ai_vs_nonai_lens_doc_weighted.csv")

# Panel A
_A = _clusters.sort_values("ai_minus_nonai", ascending=False).reset_index(drop=True)
_A["AI share (%)"] = (_A["ai_share"] * 100).round(1)
_A["Non-AI share (%)"] = (_A["nonai_share"] * 100).round(1)
_A["Difference (pp)"] = (_A["ai_minus_nonai"] * 100).round(1)
_A["Direction"] = _A["Difference (pp)"].apply(lambda x: "AI over-indexed" if x > 0 else "AI under-indexed")
_panel_a = _A[["cluster_id", "cluster_label", "AI share (%)", "Non-AI share (%)", "Difference (pp)", "Direction"]]
_panel_a = _panel_a.rename(columns={"cluster_id": "Cluster ID", "cluster_label": "Cluster label"})

# Panel B
_B = _lenses_tech.merge(
    _lenses_doc[["lens", "non_ai_doc_weighted_share", "ai_minus_doc_weighted"]],
    left_on="framing_lens", right_on="lens", how="left"
).drop(columns=["lens"])
_B = _B.sort_values("ai_minus_nonai", ascending=False).reset_index(drop=True)
_B["AI share (%)"] = (_B["ai_share"] * 100).round(1)
_B["Non-AI share, tech-weighted (%)"] = (_B["nonai_share"] * 100).round(1)
_B["Non-AI share, doc-weighted (%)"] = (_B["non_ai_doc_weighted_share"] * 100).round(1)
_B["Difference, tech-weighted (pp)"] = (_B["ai_minus_nonai"] * 100).round(1)
_B["Difference, doc-weighted (pp)"] = (_B["ai_minus_doc_weighted"] * 100).round(1)
_panel_b = _B[["framing_lens", "AI share (%)",
               "Non-AI share, tech-weighted (%)", "Difference, tech-weighted (pp)",
               "Non-AI share, doc-weighted (%)", "Difference, doc-weighted (pp)"]]
_panel_b = _panel_b.rename(columns={"framing_lens": "Framing lens"})

print("=== Table 2 Panel A: Cluster-level AI distinctiveness ===")
print(_panel_a.to_string(index=False))
print("\n=== Table 2 Panel B: Lens-level AI distinctiveness (both baselines) ===")
print(_panel_b.to_string(index=False))

# Save: CSV with both panels stacked
import io as _io
_csv_buf = _io.StringIO()
_csv_buf.write("Panel A: Cluster-level AI distinctiveness\n")
_panel_a.to_csv(_csv_buf, index=False)
_csv_buf.write("\n\nPanel B: Lens-level AI distinctiveness (both baselines)\n")
_panel_b.to_csv(_csv_buf, index=False)
(PAPER_ASSETS / "table2_ai_distinctiveness.csv").write_text(_csv_buf.getvalue())

# Save: Excel with two sheets
with pd.ExcelWriter(PAPER_ASSETS / "table2_ai_distinctiveness.xlsx") as _w:
    _panel_a.to_excel(_w, sheet_name="Panel A - Clusters", index=False)
    _panel_b.to_excel(_w, sheet_name="Panel B - Lenses", index=False)

print(f"\nSaved: {PAPER_ASSETS / 'table2_ai_distinctiveness.xlsx'}")
print(f"       {PAPER_ASSETS / 'table2_ai_distinctiveness.csv'}")


This cell generates a horizontal bar chart that visually compares the top 20 distinctive concern clusters related to AI dialogues against non-AI dialogues, highlighting both the most over-indexed and under-indexed concerns. By illustrating the differences in concern framings, this analysis is crucial for understanding how public perceptions of AI differ from those of other technologies, thereby informing future discussions and policy-making in the realm of AI and public dialogue.

In [ ]:
# @title (paper) Figure 3: AI vs non-AI distinctive clusters
# Paired horizontal bars showing the top 20 AI-distinctive concern clusters
# (10 most over-indexed, 10 most under-indexed) with AI share and non-AI
# share visible side by side.

import numpy as _np
import matplotlib.pyplot as _plt

PAPER_ASSETS = OUTPUT_FOLDER / "paper_assets"
PAPER_ASSETS.mkdir(exist_ok=True)

_dc = pd.read_csv(OUTPUT_FOLDER / "ai_distinctive_concerns.csv")
_dc = _dc.sort_values("ai_minus_nonai", ascending=True).reset_index(drop=True)
_dc["ai_pct"] = _dc["ai_share"] * 100
_dc["nonai_pct"] = _dc["nonai_share"] * 100
_dc["diff_pp"] = _dc["ai_minus_nonai"] * 100

_n = len(_dc)
_labels = _dc["cluster_label"].tolist()

fig, ax = _plt.subplots(figsize=(10.5, max(7, _n * 0.42)))
_bar_h = 0.36
_y = _np.arange(_n)
_ai_color = "#1f77b4"
_nonai_color = "#8a8a8a"

ax.barh(_y - _bar_h/2, _dc["ai_pct"], height=_bar_h, color=_ai_color,
        label="AI", edgecolor="white", linewidth=0.5)
ax.barh(_y + _bar_h/2, _dc["nonai_pct"], height=_bar_h, color=_nonai_color,
        label="Non-AI (technology-weighted)", edgecolor="white", linewidth=0.5)

ax.set_yticks(_y)
ax.set_yticklabels(_labels, fontsize=9.5)
ax.set_xlabel("Share of concern phrases (%)", fontsize=10)
ax.tick_params(axis="x", labelsize=9)

_xmax = max(_dc["ai_pct"].max(), _dc["nonai_pct"].max()) * 1.18
ax.set_xlim(0, _xmax)

for i, row in _dc.iterrows():
    diff = row["diff_pp"]
    sign = "+" if diff > 0 else ""
    color_diff = _ai_color if diff > 0 else "#666666"
    weight = "bold" if abs(diff) > 1.5 else "normal"
    longest = max(row["ai_pct"], row["nonai_pct"])
    ax.annotate(f"Δ {sign}{diff:.1f}pp", xy=(longest + 0.15, i),
                ha="left", va="center", fontsize=9, color=color_diff, fontweight=weight)

_n_under = (_dc["diff_pp"] < 0).sum()
ax.axhline(_n_under - 0.5, color="#bbbbbb", linewidth=0.7, linestyle="--", zorder=0)

ax.text(_xmax * 0.97, _n_under + (_n - _n_under) / 2 - 0.5, "AI over-indexed",
        ha="right", va="center", fontsize=10, color=_ai_color,
        fontweight="bold", alpha=0.55, rotation=90)
ax.text(_xmax * 0.97, _n_under / 2 - 0.5, "AI under-indexed",
        ha="right", va="center", fontsize=10, color="#666666",
        fontweight="bold", alpha=0.55, rotation=90)

ax.set_title("AI vs non-AI concern profile: 20 most distinctive clusters\n"
             "Top 10 most AI-over-indexed and top 10 most AI-under-indexed",
             fontsize=11, pad=10)
ax.legend(loc="lower right", fontsize=9, frameon=True, framealpha=0.95)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#cccccc")
ax.grid(axis="x", linestyle=":", linewidth=0.5, color="#dddddd", alpha=0.7, zorder=0)
ax.set_axisbelow(True)

fig.text(0.02, 0.005,
         "Δ = AI share minus non-AI share (percentage points). "
         "Non-AI baseline is technology-weighted: each non-AI technology contributes equally.",
         fontsize=8, color="#555555", ha="left")

_plt.tight_layout(rect=[0, 0.03, 1, 1])
_plt.savefig(PAPER_ASSETS / "figure3_ai_distinctive_clusters.png",
             dpi=200, bbox_inches="tight", facecolor="white")
_plt.show()
print(f"Saved: {PAPER_ASSETS / 'figure3_ai_distinctive_clusters.png'}")
